# First is joining all the hierarchical data and then randomly samplying 500 for me to hand annotate into categories

In [ ]:
import pandas as pd
import geopandas as gpd

In [ ]:
one = gpd.read_file('/Users/kaelananderson/Documents/HDMA/311 Data/data/2018/hierarchical_data_2018.geojson')
two = gpd.read_file('/Users/kaelananderson/Documents/HDMA/311 Data/data/2019/hierarchical_data_2019.geojson')
three = gpd.read_file('/Users/kaelananderson/Documents/HDMA/311 Data/data/2020/hierarchical_data_2020.geojson')
four = gpd.read_file('/Users/kaelananderson/Documents/HDMA/311 Data/data/2021/hierarchical_data_2021.geojson')
five = gpd.read_file('/Users/kaelananderson/Documents/HDMA/311 Data/data/2022/hierarchical_data_2022.geojson')
six = gpd.read_file('/Users/kaelananderson/Documents/HDMA/311 Data/data/2023/hierarchical_data_2023.geojson')

In [ ]:
# Concatenate all dataframes vertically
merged_df = pd.concat([one, two, three, four, five, six], ignore_index=True)

# Verify the merge
print(f"Total number of records: {len(merged_df)}")
print("\nDataframe columns:")

In [ ]:
merged_df.head()

In [ ]:
# Count null values in each column
null_counts = merged_df.isnull().sum()
print("Null values in each column:")
print(null_counts)

In [ ]:
# Take a random sample of 500 records from merged_df
sample_df = merged_df.sample(n=500, random_state=42)

In [ ]:
sample_df.head()

In [ ]:
sample_df.to_csv('/Users/kaelananderson/Documents/HDMA/311 Data/data/500_sample_df.csv', index=False)

# Above was sampling the entire 311 hiearchical dataset I had before. The file was then exported. 

### I used label studio to annotate the sampled data. For eaach I hand annotated their cateogries based on the descriptions. Once annotation was finished, below I read in the annotated dataset, clean the data to match columns and types of the original for easy comparison, and obtain accuracy and F-1 scores.

In [ ]:
annotated = pd.read_csv("/Users/kaelananderson/Documents/HDMA/311 Data/data/annotated_500_sample.csv")

In [ ]:
annotated.head()

In [ ]:
cols_to_drop = [
    "id",
    "updated_at",
    "lead_time",
    "Abandoned Camp",
    "Abandoned Vehicle",
    "Active Camp",
    "Camp",
    "Daily Activities",
    "Individual",
    "Matches Homeless/Encampment Filter",
    "Mental Health Crisis",
    "Neutral Presence",
    "Transient Camp",
    "Using Drugs",
    "Vehicle",
    "annotation_id",
    "annotator",
    "created_at",
    "geometry",
    "record_id"
]

annotated.drop(cols_to_drop, axis=1, inplace=True)

In [ ]:
annotated.columns

In [ ]:
annotated.head()

In [ ]:
# Get unique values from each column
columns_to_process = ['overall', 'parent', 'camp_sub', 'individual_sub', 'vehicle_sub']

for col in columns_to_process:
    # Get unique values, dropping NA/None values
    unique_values = annotated[col].dropna().unique()
    
    # Create binary columns for each unique value
    for value in unique_values:
        # Create a clean column name by removing special characters and spaces
        clean_name = f"{col}_{value}".replace(" ", "_").replace("/", "_")
        # Create boolean column instead of int
        annotated[clean_name] = (annotated[col] == value)

# Display the new columns
print("New boolean columns created:", [col for col in annotated.columns if any(base in col for base in ['overall_', 'parent_', 'camp_sub_', 'individual_sub_', 'vehicle_sub_'])])

In [ ]:
annotated.head()

In [ ]:
annotated.drop(columns=["camp_sub", "individual_sub", "overall", "parent", "vehicle_sub"], inplace=True)
annotated.head()

In [ ]:
sample_df.columns

In [ ]:
# Rename the column from 'overall_Homeless' to 'Matches Homeless/Encampment Filter'
annotated.rename(columns={
    'overall_Homeless': 'Matches Homeless/Encampment Filter',

    'parent_Camp': 'Camp',
    'camp_sub_Active_Camp': 'Active Camp',
    'camp_sub_Abandoned_Camp': 'Abandoned Camp',
    'camp_sub_Transient_Camp': 'Transient Camp',

    'parent_Vehicle': 'Vehicle',
    'vehicle_sub_Abandoned_Vehicle': 'Abandoned Vehicle',

    'parent_Individual': 'Individual',
    'individual_sub_Neutral_Presence': 'Neutral Presence',
    'individual_sub_Daily_Activities': 'Daily Activities',
    'individual_sub_Mental_Health_Crisis': 'Mental Health Crisis',
    'individual_sub_Using_Drugs': 'Using Drugs',

    }, 
inplace=True)

In [ ]:
annotated.head()

In [ ]:
sample_df.columns[3:-1]

In [ ]:
annotated.columns

In [ ]:
# Reorder columns to match the specified order
desired_order = ['date', 'description', 'Matches Homeless/Encampment Filter', 'Camp', 'Vehicle', 'Individual',
                'Active Camp', 'Abandoned Camp', 'Transient Camp', 'Abandoned Vehicle',
                'Mental Health Crisis', 'Neutral Presence', 'Daily Activities', 'Using Drugs']

annotated = annotated.reindex(columns=desired_order)

In [ ]:
sample_df.drop(columns=["geometry", "record_id", "date", "description"], inplace=True)

In [ ]:
annotated.drop(columns=["date", "description"], inplace=True)

In [ ]:
# Get column names and dtypes for both dataframes
annotated_info = pd.DataFrame({'Type': annotated.dtypes})
sample_info = pd.DataFrame({'Type': sample_df.dtypes})

# Compare the columns
comparison = pd.DataFrame({
    'Annotated Type': annotated_info['Type'],
    'Sample Type': sample_info['Type']
})

print("Column comparison:")
print(comparison)

# Check for any columns that exist in one but not the other
annotated_cols = set(annotated.columns)
sample_cols = set(sample_df.columns)

print("\nColumns only in annotated:", annotated_cols - sample_cols)
print("Columns only in sample_df:", sample_cols - annotated_cols)

In [ ]:
sample_df.reset_index(inplace=True)
sample_df.head()

In [ ]:
annotated.reset_index(inplace=True)
sample_df.head()

# Accuracies

## Column-wise accuracy

Find the accuracies of each column, independent of each other

In [ ]:
common_columns = annotated.columns.intersection(sample_df.columns)

# Compute Column-wise Accuracy
column_accuracy = (annotated[common_columns] == sample_df[common_columns]).mean()
print("Column-wise Accuracy:")
print(column_accuracy)

## Hierarchicial Accuracy

### Hierarchical (Combined) Accuracy Explanation

This code calculates a combined accuracy metric that takes into account both the parent category and its corresponding sub-category for each record. The process is as follows:

1. **Identify the Parent Category:**
   - The `get_parent` function checks a row to see which parent category is selected. It assumes that only one of the columns `'Camp'`, `'Individual'`, or `'Vehicle'` is non-null.
   - It returns the name of the first non-null parent it finds.

2. **Extract the Sub-Category:**
   - The `get_subcategory` function uses the identified parent to extract the sub-category value from the appropriate columns:
     - For **Camp**: It checks `'Active Camp'`, `'Abandoned Camp'`, and `'Transient Camp'`.
     - For **Individual**: It checks `'Using Drugs'`, `'Neutral Presence'`, `'Daily Activities'`, and `'Mental Health Crisis'`.
     - For **Vehicle**: It checks `'Abandoned Vehicle'`.
   - It returns the first non-null sub-category value found for that parent.

3. **Compare Ground Truth and Prediction:**
   - The `hierarchical_accuracy` function:
     - Determines the parent for both the ground truth and predicted rows.
     - If the parent categories do not match, the record is immediately marked incorrect.
     - If the parents match, it then extracts and compares the corresponding sub-categories.
     - A record is considered correct only if both the parent and its sub-category are the same between ground truth and prediction.

4. **Overall Accuracy Calculation:**
   - The code iterates over all records (using `iterrows()` on the ground truth (`annotated`) and prediction (`sample_df`) DataFrames).
   - For each pair of rows, it uses `hierarchical_accuracy` to determine if the prediction is correct.
   - The combined correct predictions are summed and then divided by the total number of records to compute the hierarchical (combined) accuracy.

*This approach ensures that the prediction is only considered correct when it identifies the correct parent category and the correct sub-category, reflecting the hierarchical nature of the dataset.*


In [ ]:
###############################################################################
# Hierarchical (Combined) Accuracy
#
# For each record:
# 1. Determine which parent category is selected by checking which of 
#    'Camp', 'Individual', or 'Vehicle' is non-null.
# 2. If the parent's value matches between ground truth and prediction, 
#    extract the corresponding sub-category.
#    - For Camp: 'Active Camp', 'Transient Camp', or 'Abandoned Camp'
#    - For Individual: 'Using Drugs', 'Neutral Presence', 'Daily Activities', or 'Mental Health Crisis'
#    - For Vehicle: 'Abandoned Vehicle'
# 3. The record is considered correct only if both the parent and the 
#    corresponding sub-category match.
###############################################################################


"""Determine the parent category from a row (assumes only one is non-null)."""
def get_parent(row):

    if pd.notna(row['Camp']):
        return 'Camp'
    elif pd.notna(row['Individual']):
        return 'Individual'
    elif pd.notna(row['Vehicle']):
        return 'Vehicle'
    else:
        return None

"""Extract the sub-category from the row based on the parent category."""
def get_subcategory(row, parent):

    if parent == 'Camp':
        if pd.notna(row['Active Camp']):
            return row['Active Camp']
        elif pd.notna(row['Abandoned Camp']):
            return row['Abandoned Camp']
        elif pd.notna(row['Transient Camp']):
            return row['Transient Camp']
        else:
            return None
    elif parent == 'Vehicle':
        return row['Abandoned Vehicle'] if pd.notna(row['Abandoned Vehicle']) else None
    elif parent == 'Individual':
        if pd.notna(row['Using Drugs']):
            return row['Using Drugs']
        elif pd.notna(row['Neutral Presence']):
            return row['Neutral Presence']
        elif pd.notna(row['Daily Activities']):
            return row['Daily Activities']
        elif pd.notna(row['Mental Health Crisis']):
            return row['Mental Health Crisis']
        else:
            return None
    else:
        return None

def hierarchical_accuracy(row_gt, row_pred):
    """Return True if both the parent and corresponding sub-category match."""

    parent_gt = get_parent(row_gt)
    parent_pred = get_parent(row_pred)
    if parent_gt != parent_pred:
        return False
    sub_gt = get_subcategory(row_gt, parent_gt)
    sub_pred = get_subcategory(row_pred, parent_pred)
    return sub_gt == sub_pred

# Compute Hierarchical (Combined) Accuracy:
combined_correct = 0
total_records = len(annotated)
for (_, row_gt), (_, row_pred) in zip(annotated.iterrows(), sample_df.iterrows()):
    if hierarchical_accuracy(row_gt, row_pred):
        combined_correct += 1
hierarchical_acc = combined_correct / total_records
print("Hierarchical (Combined) Accuracy:", hierarchical_acc)



## Column-wise F-1 scores

## Hierarchical micro F-1 Score

### Code Explanation

**1. Individual F1 Score Calculation:**
- **Function `compute_f1(true_series, pred_series)`:**
  - Counts:
    - **TP (True Positives):** Both true and predicted values are `True`.
    - **FP (False Positives):** True value is `False` but predicted value is `True`.
    - **FN (False Negatives):** True value is `True` but predicted value is `False`.
  - Computes:
    - **Precision:** `TP / (TP + FP)`
    - **Recall:** `TP / (TP + FN)`
    - **F1 Score:** Harmonic mean of precision and recall.
- **Loop:** Iterates over each category in `common_columns`, calculates its F1 score using `compute_f1`, and prints it.

**2. Micro F1 Score Calculation:**
- **Aggregate Counts:** Sums TP, FP, and FN for all categories.
- **Compute Micro Metrics:**
  - **Micro Precision:** `total_tp / (total_tp + total_fp)`
  - **Micro Recall:** `total_tp / (total_tp + total_fn)`
- **Micro F1 Score:** Calculated as the harmonic mean of micro precision and micro recall.

*This approach gives equal F1 scores for individual categories and an overall metric (micro F1) that weights more frequent (e.g., parent) categories more heavily, which is beneficial for the nature of the hierarchical dataset.*


In [ ]:
def compute_f1(true_series, pred_series):
    """
    Computes the F1 score given two boolean pandas Series.
    """
    tp = ((true_series == True) & (pred_series == True)).sum()
    fp = ((true_series == False) & (pred_series == True)).sum()
    fn = ((true_series == True) & (pred_series == False)).sum()
    
    # Avoid division by zero
    if tp + fp == 0 or tp + fn == 0:
        return 0.0
    
    precision = tp / (tp + fp)
    recall = tp / (tp + fn)
    
    if precision + recall == 0:
        return 0.0
    return 2 * precision * recall / (precision + recall)

# Calculate individual F1 scores for each category
individual_f1 = {}
print("Individual F1 Scores:")
for cat in common_columns:
    f1 = compute_f1(annotated[cat], sample_df[cat])
    individual_f1[cat] = f1
    print(f"  {cat}: {f1:.3f}")

# Now compute micro F1 across all specified categories
total_tp = 0
total_fp = 0
total_fn = 0

for cat in common_columns:
    tp = ((annotated[cat] == True) & (sample_df[cat] == True)).sum()
    fp = ((annotated[cat] == False) & (sample_df[cat] == True)).sum()
    fn = ((annotated[cat] == True) & (sample_df[cat] == False)).sum()
    
    total_tp += tp
    total_fp += fp
    total_fn += fn

if total_tp + total_fp == 0 or total_tp + total_fn == 0:
    micro_f1 = 0.0
else:
    micro_precision = total_tp / (total_tp + total_fp)
    micro_recall = total_tp / (total_tp + total_fn)
    if micro_precision + micro_recall == 0:
        micro_f1 = 0.0
    else:
        micro_f1 = 2 * micro_precision * micro_recall / (micro_precision + micro_recall)

print(f"\nMicro F1 Score across categories: {micro_f1:.3f}")
